# Genjutsu Studio: بديل Higgsfield المفتوح بضغطة زرار واحدة

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f12mano/video-depth-colab/blob/main/video_depth_colab.ipynb)
[![Higgsfield V2V](https://img.shields.io/badge/Higgsfield_V2V-Reverse_Engineered-black?style=flat&color=32D74B)](https://github.com/f12mano/video-depth-colab/tree/main)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/f12mano/video-depth-colab/tree/main)
[![License: Apache 2.0](https://img.shields.io/badge/License-Apache_2.0-green.svg)](https://opensource.org/licenses/Apache-2.0)

<p align="center">
  <img src="https://raw.githubusercontent.com/f12mano/Genjustsu-Open-Source-Workflow/main/examples/previews/workflow.gif" width="30%" alt="Workflow" />
  <img src="https://raw.githubusercontent.com/f12mano/Genjustsu-Open-Source-Workflow/main/examples/previews/example-1.gif" width="33%" alt="Example 1" />
  <img src="https://raw.githubusercontent.com/f12mano/Genjustsu-Open-Source-Workflow/main/examples/previews/example-2.gif" width="33%" alt="Example 2" />
</p>

معاينة حية للنتائج قبل وبعد استبدال الشخصية وتوجيه الحركة بالمللي

سر بايبلاين Genjutsu اللي اتعمل له هندسة عكسية من Higgsfield لاستبدال أي شخصية في الفيديو بدقة حتمية على Omni و Seedance و Kling
بيجمع بين عزل الكتل بـ Meta SAM 2.1 والعمق بـ ByteDance VDA وتتبع ملامح الوش بـ Google MediaPipe في رندر سحابي مجاني 100%

- كارت الشاشة المطلوب: T4 GPU مجاني على Google Colab
- الواجهة: ويب Gradio بتفتح برابط خاص على طول
- المخرجات: فيديو التوجيه الحركي MP4 مع طبقة الـ Depth الحقيقية

---

### أصل الفكرة وشكر واجب
الفكرة دي مبنية على الهندسة العكسية العبقرية اللي عملها المطور Sirio Berati في مشروعه [Genjutsu Workflow](https://github.com/f12mano/Genjustsu-Open-Source-Workflow) وشرحها في الفيديو بتاعه [Dear Higgsfield, I reverse engineered you in 12 hours](https://www.youtube.com/watch?v=d6spCMcfkew)
هو أول واحد أثبت إن الدمج بين خريطة العمق وشبكة ملامح الوش مع الحفاظ على خلفية الأوضة هو السر الحقيقي اللي بيمنع موديلات الفيديو من الهلوسة

مقارنة معمارية بين طريقتنا وطريقة سيريو الأصلية:

طريقة سيريو Genjutsu الأصلية:
- الميكانيزم: سيرفر محلي FastApi مربوط بـ Webhooks ونفق Cloudflare وبيرحل الرندر لخدمات Replicate و Enhancor
- النقط الإيجابية: بتشتغل على أي جهاز ضعيف زي ماك بوك أير من غير كارت شاشة خالص لأن الرندر برة / مجهزة للإنتاج التجاري المتكامل مع APIs توليد الفيديو
- التحديات: خطوات تسطيب كتير ومحتاجة بايثون ومكتبات صوت وفتح حسابات APIs وشحن رصيد وتظبيط مفاتيح سرية في ملف البيئة

طريقتنا One-Click Colab Runner:
- الميكانيزم: بيئة سحابية واحدة على Google Colab بتشغل موديلات Meta و ByteDance و Google محلياً على كارت T4 سحابي مباشر
- النقط الإيجابية: بضغطة زرار واحدة من غير أي تسطيب على جهازك ولا فتح حسابات ولا ربط مفاتيح سرية / بتطلعلك فيديو التوجيه الحركي كامل قدام عينك في دقيقة
- التحديات: جلسة كولاب مؤقتة وبتقفل لو سبتها فاضية / بتسلّمك الفيديو الإرشادي عشان ترفعه بنفسك للمنصة بدل ما تبعته كود أوتوماتيك

---

### خطوات التشغيل
1. اتأكد إن نوع الحوسبة مضبوط على كارت T4 GPU من قائمة Runtime
2. شغّل كل الخلايا ورا بعض من Runtime وبعدها Run all أو دوس Ctrl + F9
3. انزل للخلية رقم 4 وافتح الرابط اللي هيظهرلك

---
الموديلات الأصلية: [Video Depth Anything](https://github.com/DepthAnything/Video-Depth-Anything) / [SAM 2.1](https://github.com/facebookresearch/sam2) / [MediaPipe Face Landmarker](https://developers.google.com/mediapipe/solutions/vision/face_landmarker)


In [ ]:
# [1] Check Allocated Cloud GPU
!nvidia-smi

In [ ]:
# [2] Clone Video-Depth-Anything repository and install dependencies
import os

if not os.path.exists('/content/Video-Depth-Anything'):
    !git clone https://github.com/DepthAnything/Video-Depth-Anything.git /content/Video-Depth-Anything

%cd /content/Video-Depth-Anything

# Install dependencies with modern compatibility
!pip install -q gradio imageio[ffmpeg] opencv-python matplotlib tqdm easydict einops mediapipe hydra-core omegaconf
!pip uninstall -y xformers || true

# Install Meta official SAM 2 without heavy CUDA extensions
!pip install -q git+https://github.com/facebookresearch/sam2.git --no-build-isolation


In [ ]:
# [3] Download Model Weights (Small: ~56MB | Large: ~1.4GB)
import os

os.makedirs('/content/Video-Depth-Anything/checkpoints', exist_ok=True)
%cd /content/Video-Depth-Anything/checkpoints

# Download Small model (Fast & Safe for T4 GPU)
if not os.path.exists('video_depth_anything_vits.pth'):
    print("[+] Downloading Video-Depth-Anything-Small (vits)...")
    !wget -q --show-progress https://huggingface.co/depth-anything/Video-Depth-Anything-Small/resolve/main/video_depth_anything_vits.pth

# Download Large model (Optional for maximum fidelity)
if not os.path.exists('video_depth_anything_vitl.pth'):
    print("[+] Downloading Video-Depth-Anything-Large (vitl)...")
    !wget -q --show-progress https://huggingface.co/depth-anything/Video-Depth-Anything-Large/resolve/main/video_depth_anything_vitl.pth

%cd /content/Video-Depth-Anything
print("[*] Ready to launch!")

In [ ]:
# [4] Fetch latest Colab Web UI and Launch with Public Link
%cd /content/Video-Depth-Anything

# Download the latest runner script directly from GitHub repo
!wget -q -O colab_app.py https://raw.githubusercontent.com/f12mano/video-depth-colab/main/colab_app.py

# Launch Gradio Web UI (Click the generated https://xxxx.gradio.live link)
!python colab_app.py